# Colab launcher (thin)

Run the cells top to bottom. This notebook only mounts Drive, gets the repo at a
pinned `REF`, builds the cache, runs the tests, and starts `src.train`.

- First pass: leave `SMOKE = True` for a GPU pipeline check.
- Real runs: set `SMOKE = False` and choose `CONFIG`.
- `RUN_SUFFIX` is appended to `run.name`, so a different epoch count, seed or
  batch size never reuses (and resumes into) the previous run folder.
- `GRAD_ACCUM` overrides `train.grad_accum_steps`, to keep the effective batch
  size when `BATCH_SIZE` has to drop on a T4.
- `SEED` overrides `run.seed` and adds `_seed<N>` to the run name, so seeds
  42, 43 and 44 of one config never share a folder.
- After training, one cell evaluates the held-out test split and appends a row
  to `RESULTS_CSV` on Drive; the last cell aggregates all seeds into one table.
- The training cell draws a live `tqdm` epoch bar from the child's log lines
  and prints everything else above it; set `RAW_STREAM = True` to print the
  raw stream instead.
- `REF` is the commit, tag or branch the repo cell checks out. A commit hash is
  the most reproducible choice; it is recorded in `run_info.json`.
- Never commit the cache, run folders, checkpoints, or your token.

Resume: rerun the training cell after a disconnect with the **same** `REF`,
`CONFIG`, `RUNS_DIR` and `RUN_SUFFIX`. It continues from
`<RUNS_DIR>/<run name>/last.pt`.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/awdtyo/Polyp-Segmentation.git"
DRIVE_ZIP = "/content/drive/MyDrive/kvasir-seg.zip"
RUNS_DIR = "/content/drive/MyDrive/polyp-runs"
CONFIG = "configs/exp001_convnext_base_baseline.yaml"
SMOKE = True          # set False for a real run
EPOCHS = None         # e.g. 5 for the first real-run probe
BATCH_SIZE = None     # e.g. 2 on a T4 if batch 8 runs out of memory
GRAD_ACCUM = None     # e.g. 2 to keep the effective batch size when BATCH_SIZE drops
SEED = None           # e.g. 43; sets run.seed and appends _seed43 to run.name
RAW_STREAM = False    # True: print child output untouched, no tqdm bar
RUN_SUFFIX = ""       # e.g. "_e5"; appended to run.name so folders never collide
RESULTS_CSV = f"{RUNS_DIR}/results.csv"   # one row per run/split, kept on Drive
REF = "main"          # commit hash, tag or branch to check out
REPO_DIR = Path("/content/polyp")


## 1. Mount Drive and get the repo

`GH_TOKEN` is read from Colab Secrets and is never printed. Add it with the key
icon in the left sidebar (scope: this notebook). If the repo is public you can
still use a token; it only needs read access.

The repo is fetched and then **checked out at `REF`** (a commit, tag or branch)
instead of a plain `git pull`, so a rerun cannot silently pick up newer code
than the one recorded in `run_info.json`. A commit hash is the most
reproducible choice.


In [ ]:
import subprocess

from google.colab import drive, userdata

drive.mount("/content/drive")

token = userdata.get("GH_TOKEN")
if not token:
    raise RuntimeError(
        "Missing GH_TOKEN. Add it in Colab Secrets (key icon) and enable it for this notebook."
    )


def _auth_url(url: str, secret: str) -> str:
    """Insert a token into an https clone URL without storing it anywhere."""
    if not url.startswith("https://"):
        raise ValueError("REPO_URL must be an https URL")
    return url.replace("https://", f"https://x-access-token:{secret}@", 1)


def _git(args: list[str], check: bool = True) -> subprocess.CompletedProcess:
    """Run git and raise a token-free message on failure (unless check=False)."""
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if check and result.returncode != 0:
        message = (result.stderr or result.stdout).replace(token, "***")
        raise RuntimeError(f"git failed: {message}")
    return result


if (REPO_DIR / ".git").is_dir():
    _git(["-C", str(REPO_DIR), "fetch", "origin", "--tags", "--prune"])
else:
    _git(["clone", _auth_url(REPO_URL, token), str(REPO_DIR)])
    _git(["-C", str(REPO_DIR), "fetch", "origin", "--tags", "--prune"])

# Check out REF. For a branch, reset it to the fetched remote branch so REF is
# exactly what runs; for a tag or commit hash, check it out directly (detached).
has_remote_branch = (
    _git(["-C", str(REPO_DIR), "rev-parse", "--verify", "--quiet", f"origin/{REF}"], check=False).returncode
    == 0
)
if has_remote_branch:
    checkout = _git(["-C", str(REPO_DIR), "checkout", "-B", REF, f"origin/{REF}"], check=False)
else:
    checkout = _git(["-C", str(REPO_DIR), "checkout", REF], check=False)
if checkout.returncode != 0:
    message = (checkout.stderr or checkout.stdout).replace(token, "***")
    raise RuntimeError(f"Could not check out REF={REF!r}: {message}")

# Do not leave the token in .git/config.
subprocess.run(
    ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL],
    check=True,
)
del token
head = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
print(f"repo ready at {head} (REF={REF})")


## 2. Install only missing Python packages

Torch is already installed on Colab and is deliberately not touched.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "timm": "timm",
    "albumentations": "albumentations",
    "yaml": "pyyaml",
    "pytest": "pytest",
    "tqdm": "tqdm",
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all required Python packages are already present")


## 3. Unzip the dataset and find `images/` + `masks/`

The folder is discovered by searching for a directory that contains both
`images/` and `masks/`; its path is never hardcoded.


In [ ]:
import shutil
import zipfile

if SMOKE:
    DATA_ROOT = None
    print("SMOKE: synthetic data only; skipping the Drive zip.")
else:
    DATA_DIR = Path("/content/data")
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True)
    with zipfile.ZipFile(DRIVE_ZIP) as archive:
        archive.extractall(DATA_DIR)

    candidates = [
        path
        for path in DATA_DIR.rglob("*")
        if path.is_dir() and (path / "images").is_dir() and (path / "masks").is_dir()
    ]
    if not candidates:
        raise FileNotFoundError(
            "No folder with both images/ and masks/ inside the zip. "
            "Check the archive layout or DRIVE_ZIP."
        )
    DATA_ROOT = max(candidates, key=lambda path: sum(1 for _ in (path / "images").iterdir()))
    n_images = sum(1 for _ in (DATA_ROOT / "images").iterdir())
    if n_images != 1000:
        print(f"warning: found {n_images} images, expected 1000")
    print("data root:", DATA_ROOT, "images:", n_images)


## 4. Build the npz cache

The cache lives in `/content`, outside git, and is rebuilt at the start of every
session from the zip on Drive. `train.py` records its SHA-256 in `run_info.json`.


In [ ]:
import sys

if SMOKE:
    CACHE_PATH = None
    print("SMOKE: skipping the cache build.")
else:
    CACHE_PATH = Path("/content/cache/kvasir_seg_256.npz")
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            sys.executable,
            str(REPO_DIR / "scripts" / "build_cache.py"),
            "--config",
            str(REPO_DIR / CONFIG),
            "--root",
            str(DATA_ROOT),
            "--out",
            str(CACHE_PATH),
        ],
        cwd=REPO_DIR,
        check=True,
    )


## 5. Sanity check

Runs the test suite (no real data, no pretrained weights, no full training).
A failure prints a warning but does not stop the notebook.


In [ ]:
import subprocess
import sys

tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
    check=False,
)
output = "\n".join(part for part in (tests.stdout, tests.stderr) if part).strip()
print("pytest output (last 20 lines):")
print("\n".join(output.splitlines()[-20:]))
if tests.returncode == 0:
    print("pytest passed.")
else:
    print(
        f"WARNING: pytest failed with exit code {tests.returncode}; "
        "training will still be allowed to proceed."
    )


## 6. Train

The committed config is copied to `/content/config_session.yaml` with absolute
`data.root`, `data.cache_path` and `run.out_dir`, then passed to `src.train`.
This keeps the repository config portable while putting checkpoints on Drive.

`RUN_SUFFIX` is appended to `run.name` (after the config is loaded), `SEED`
overrides `run.seed` and appends `_seed<N>`, and
`GRAD_ACCUM` overrides `train.grad_accum_steps`. Use a suffix for every probe
with a different epoch count or seed: a resumed run keeps the folder name, and
resuming a short probe with a different epoch count would change the cosine
schedule.

`--smoke` is a pipeline test on synthetic data (tiny model, 64 px, 3 steps). Its
numbers are not results.

The child process is started with `subprocess.Popen` and its output is streamed
line by line (`-u`, `PYTHONUNBUFFERED=1`, `stderr=STDOUT`). Without that,
Python block-buffers stdout into the pipe and the traceback of a failed run can
be lost. The cell prints the exit code and raises only when it is non-zero.
The command is `python -m src.train` (not `python src/train.py`) so the `src`
package is importable; the launcher is otherwise identical to the CLI.

Output is shown as a live `tqdm` bar over epochs. The bar advances on every
`epoch N: train ... val ... dice ... iou ...` line, and its postfix carries
train/val loss and val Dice/IoU; every other line (including the epoch line
itself) is printed above the bar with `tqdm.write`. A resumed run starts the
bar at the first epoch still to run, read from the `resumed from ... at epoch
N` line. Set `RAW_STREAM = True` to skip the bar and pass the child output
through untouched, for example when capturing the log to a file.


In [ ]:
from pathlib import Path

import os
import re
import subprocess
import sys

import torch
import yaml

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Only --smoke is allowed on CPU.")

cfg = yaml.safe_load((REPO_DIR / CONFIG).read_text(encoding="utf-8"))
if not SMOKE:
    cfg.setdefault("data", {})
    cfg["data"]["root"] = str(DATA_ROOT)
    cfg["data"]["cache_path"] = str(CACHE_PATH)
cfg.setdefault("run", {})
cfg["run"]["out_dir"] = RUNS_DIR
cfg["run"]["name"] = f"{cfg['run']['name']}{RUN_SUFFIX}"
if SEED is not None:
    cfg["run"]["seed"] = SEED
    cfg["run"]["name"] = f"{cfg['run']['name']}_seed{SEED}"
if EPOCHS is not None:
    cfg.setdefault("train", {})["epochs"] = EPOCHS
if BATCH_SIZE is not None:
    cfg.setdefault("train", {})["batch_size"] = BATCH_SIZE
if GRAD_ACCUM is not None:
    cfg.setdefault("train", {})["grad_accum_steps"] = GRAD_ACCUM

Path(RUNS_DIR).mkdir(parents=True, exist_ok=True)
session_config = Path("/content/config_session.yaml")
session_config.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

cmd = [sys.executable, "-u", "-m", "src.train", "--config", str(session_config)]
if SMOKE:
    cmd.append("--smoke")
print("run name:", cfg["run"]["name"])
print("run seed:", cfg["run"]["seed"])
print("training command:", " ".join(cmd))
print("rerun this cell after a disconnect to resume from last.pt", flush=True)

env = dict(os.environ, PYTHONUNBUFFERED="1")
process = subprocess.Popen(
    cmd,
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
assert process.stdout is not None
if RAW_STREAM:
    for line in process.stdout:
        print(line, end="", flush=True)
else:
    from tqdm.auto import tqdm

    total_epochs = 1 if SMOKE else int(cfg["train"]["epochs"])
    epoch_line = re.compile(
        r"^epoch\s+(\d+):\s+train\s+([\d.]+)\s+val\s+([\d.]+)"
        r"\s+dice\s+([\d.]+)\s+iou\s+([\d.]+)"
    )
    resumed_line = re.compile(r"resumed from .* at epoch (\d+)")

    bar = tqdm(total=total_epochs, desc=cfg["run"]["name"], unit="epoch", dynamic_ncols=True)
    for line in process.stdout:
        text = line.rstrip("\n")
        resumed = resumed_line.search(text)
        if resumed:
            # The child skips the epochs already in last.pt; show them as done.
            bar.update(int(resumed.group(1)))
        tqdm.write(text, file=sys.stdout)  # keep every log line above the bar
        match = epoch_line.match(text)
        if match:
            _, train_loss, val_loss, dice, iou = match.groups()
            bar.set_postfix(
                train_loss=f"{float(train_loss):.4f}",
                val_loss=f"{float(val_loss):.4f}",
                val_dice=f"{float(dice):.4f}",
                val_iou=f"{float(iou):.4f}",
            )
            bar.update(1)
    bar.close()

return_code = process.wait()
print(f"train.py exit code: {return_code}", flush=True)
if return_code != 0:
    raise RuntimeError(f"python -m src.train failed with exit code {return_code}")


## 7. Read the measured memory and timing

After the first real run, copy peak VRAM, mean epoch time and Dice into the
"Colab results" table in `PROGRESS.md`.


In [ ]:
import json
from pathlib import Path

run_name = cfg["run"]["name"] + ("_smoke" if SMOKE else "")
run_dir = Path(RUNS_DIR) / run_name
print((run_dir / "metrics.csv").read_text(encoding="utf-8"))

info = json.loads((run_dir / "run_info.json").read_text(encoding="utf-8"))
print(
    json.dumps(
        {
            key: info.get(key)
            for key in (
                "gpu_name",
                "params_total",
                "peak_gpu_mem_mb",
                "mean_epoch_time_s",
                "epochs_this_session",
            )
        },
        indent=2,
    )
)


## 8. Evaluate on the held-out test split

One row is appended to `RESULTS_CSV` on Drive for this `(run name, split)`. The
test split is evaluated **once**, at the end, and is never used for model
selection; `src.evaluate` prints an explicit warning and refuses to run if a
test id also appears in `train.txt` or `val.txt`. Re-running is refused unless
you add `--force`, which replaces the row in place instead of duplicating it.

In `SMOKE` mode the synthetic test ids are a subset of `val`, so the cell
evaluates `val` instead: the guard is what makes that necessary.


In [ ]:
import subprocess
import sys
from pathlib import Path

run_name = cfg["run"]["name"] + ("_smoke" if SMOKE else "")
run_dir = Path(RUNS_DIR) / run_name
eval_split = "val" if SMOKE else "test"

if not (run_dir / "best.pt").is_file():
    raise FileNotFoundError(f"No best.pt in {run_dir}; run the training cell first.")

Path(RESULTS_CSV).parent.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable,
    "-m",
    "src.evaluate",
    "--run_dir",
    str(run_dir),
    "--split",
    eval_split,
    "--checkpoint",
    "best",
    "--results_csv",
    str(RESULTS_CSV),
]
print("evaluate command:", " ".join(cmd), flush=True)
if eval_split == "test":
    print("TEST SPLIT: this is the only time these numbers are produced.", flush=True)
result = subprocess.run(cmd, cwd=REPO_DIR, text=True, check=False)
print(f"evaluate exit code: {result.returncode}", flush=True)
if result.returncode != 0:
    raise RuntimeError(
        f"src.evaluate failed with exit code {result.returncode}; "
        "a duplicate row needs --force, a leaky test split must be fixed."
    )
print("per-image file:", run_dir / f"per_image_{eval_split}.csv")
print("results csv:", RESULTS_CSV)


## 9. Aggregate the seeds into one table

Groups every run under `RUNS_DIR` by config (the `_seed<N>` suffix is ignored)
and writes `aggregate_results.md` / `.csv` there: mean ± std test Dice and IoU
across seeds, small-lesion Dice, parameters and mean epoch time. A config with
one seed shows `n/a` for std, never `0.0`.

For a paired comparison of two configs, add
`--compare <config A> <config B>`; it reports the per-image Dice difference
averaged over shared seeds with a fixed-seed bootstrap 95% CI. That CI describes
this fixed test set only.


In [ ]:
import subprocess
import sys
from pathlib import Path

if SMOKE:
    print("SMOKE: no test evaluations were written, so there is nothing to aggregate.")
else:
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts" / "aggregate_results.py"),
        "--runs-dir",
        RUNS_DIR,
        "--out-dir",
        RUNS_DIR,
    ]
    print("aggregate command:", " ".join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, check=False)
    print(f"aggregate exit code: {result.returncode}", flush=True)
    if result.returncode != 0:
        raise RuntimeError(f"aggregate_results.py failed with exit code {result.returncode}")
    print((Path(RUNS_DIR) / "aggregate_results.md").read_text(encoding="utf-8"))
